In [2]:
from pathlib import Path
import json

import joblib
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# In Jupyter/Colab, __file__ is not defined. We use the current working directory instead.
ROOT = Path.cwd()
MODEL_DIR = ROOT / "models"
REPORT_DIR = ROOT / "reports"

MODEL_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)


def load_data():
    dataset = load_breast_cancer(as_frame=True)
    return dataset, dataset.data, dataset.target


def build_models():
    scaled_preprocessing = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    tree_preprocessing = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ])

    return {
        "Logistic Regression": Pipeline([
            ("preprocessing", scaled_preprocessing),
            ("model", LogisticRegression(
                max_iter=5000, random_state=42
            )),
        ]),

        "Random Forest": Pipeline([
            ("preprocessing", tree_preprocessing),
            ("model", RandomForestClassifier(
                n_estimators=300,
                class_weight="balanced",
                random_state=42,
            )),
        ]),

        "HistGradientBoosting": Pipeline([
            ("preprocessing", tree_preprocessing),
            ("model", HistGradientBoostingClassifier(
                random_state=42
            )),
        ]),
    }


def main():
    dataset, X, y = load_data()

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        stratify=y,
        random_state=42,
    )

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42,
    )

    results = []
    trained_models = {}

    for name, model in build_models().items():
        scores = cross_val_score(
            model,
            X_train,
            y_train,
            cv=cv,
            scoring="f1",
            n_jobs=-1,
        )

        model.fit(X_train, y_train)
        trained_models[name] = model

        results.append({
            "model": name,
            "cv_f1_mean": scores.mean(),
            "cv_f1_std": scores.std(),
        })

    comparison = pd.DataFrame(results).sort_values(
        "cv_f1_mean", ascending=False
    )

    best_name = comparison.iloc[0]["model"]
    best_model = trained_models[best_name]

    predictions = best_model.predict(X_test)
    probabilities = best_model.predict_proba(X_test)[:, 1]

    metrics = {
        "selected_model": best_name,
        "accuracy": float(
            accuracy_score(y_test, predictions)
        ),
        "precision": float(
            precision_score(
                y_test, predictions, zero_division=0
            )
        ),
        "recall": float(
            recall_score(
                y_test, predictions, zero_division=0
            )
        ),
        "f1": float(
            f1_score(
                y_test, predictions, zero_division=0
            )
        ),
        "roc_auc": float(
            roc_auc_score(y_test, probabilities)
        ),
        "confusion_matrix": confusion_matrix(
            y_test, predictions
        ).tolist(),
        "classification_report": classification_report(
            y_test,
            predictions,
            target_names=dataset.target_names,
            output_dict=True,
            zero_division=0,
        ),
    }

    comparison.to_csv(
        REPORT_DIR / "model_comparison.csv",
        index=False,
    )

    (REPORT_DIR / "test_metrics.json").write_text(
        json.dumps(metrics, indent=2),
        encoding="utf-8",
    )

    joblib.dump(
        best_model,
        MODEL_DIR / "best_model.joblib",
    )

    print("\nModel comparison:")
    print(comparison.to_string(index=False))

    print(f"\nBest model: {best_name}")
    print("\nTest metrics:")

    for metric in [
        "accuracy", "precision", "recall", "f1", "roc_auc"
    ]:
        print(f"{metric}: {metrics[metric]:.4f}")

    print("\nConfusion matrix:")
    print(metrics["confusion_matrix"])

    print("\nSaved model and reports successfully.")


if __name__ == "__main__":
    main()


Model comparison:
               model  cv_f1_mean  cv_f1_std
 Logistic Regression    0.982544   0.007752
HistGradientBoosting    0.975528   0.010452
       Random Forest    0.970001   0.013501

Best model: Logistic Regression

Test metrics:
accuracy: 0.9825
precision: 0.9861
recall: 0.9861
f1: 0.9861
roc_auc: 0.9954

Confusion matrix:
[[41, 1], [1, 71]]

Saved model and reports successfully.
